# S&P 500 Dataset — Data Overview & Visualization

This notebook provides a complete visual EDA of `sp500_macro_master.csv`, built by `build_dataset.py`. The dataset covers 99 top S&P 500 tickers by market-cap weight over ~5 years of daily OHLCV data (2019–2024), merged with 11 macro covariates sourced from FRED and Kaggle.

**What we'll cover:**
1. Dataset shape & dtypes
2. Missing value analysis
3. Ticker & sector universe
4. Close price distributions
5. Macro covariate time series & correlations
6. Sector-level return & volatility profiles
7. Volume analysis

In [ ]:
import importlib, subprocess, sys

for pkg in ['seaborn', 'pandas', 'matplotlib', 'numpy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')
print('Environment ready.')

## 1. Library Imports & Global Style

We use seaborn's `whitegrid` theme with the `muted` palette throughout all plots. Default figure size is set to `(12, 5)` at 120 DPI for clear rendering in notebooks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

COVARIATE_COLS = [
    'DGS10', 'VIXCLS', 'UNRATE', 'CPIAUCSL', 'BAMLH0A0HYM2',
    'T10Y2Y', 'UMCSENT', 'FEDFUNDS_RATE', 'M2_MONEY_SUPPLY',
    'SOFR', 'INFLATION_RATE'
]
DATA_PATH = 'data/sp500_macro_master.csv'
print('Libraries loaded. Seaborn version:', sns.__version__)

## 2. Dataset Loading & Basic Info

We load the master CSV, parse dates, and get a structural overview of shape, dtypes, memory usage, and the date range covered by the dataset.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Shape:       {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns")
print(f"Tickers:     {df['Ticker'].nunique()}")
print(f"Sectors:     {df['Sector'].nunique()}")
print(f"Date range:  {df['Date'].min().date()} \u2192 {df['Date'].max().date()}")
print(f"Memory:      {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
print("\u2500\u2500\u2500 dtypes \u2500\u2500\u2500")
print(df.dtypes.to_string())
print()
print("\u2500\u2500\u2500 describe (price + macro) \u2500\u2500\u2500")
print(df[['Close', 'Volume'] + COVARIATE_COLS].describe().round(3).to_string())

## 3. Missing Value Analysis

We check for any remaining nulls across all columns. A clean dataset is critical before feeding data into time series foundation models — even a single NaN in the context window can cause silent errors or NaN outputs.

In [ ]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(4)
null_df     = pd.DataFrame({'count': null_counts, 'pct': null_pct})
null_df     = null_df[null_df['count'] > 0]

if null_df.empty:
    print("\u2705 Zero null values across all 17 columns.")
else:
    print(f"\u26a0\ufe0f  Null values found:\n{null_df}")

fig, ax = plt.subplots(figsize=(14, 3))
sample = df.sample(min(5000, len(df)), random_state=42).sort_index()
sns.heatmap(sample.isnull(), yticklabels=False, cbar=False,
            cmap='Reds', ax=ax)
ax.set_title('Missing Value Map (5,000-row random sample) \u2014 All White = No Nulls')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('notebooks/02_missing_values.png', bbox_inches='tight')
plt.show()

## 4. Ticker & Sector Universe

How are the 99 tickers distributed across GICS sectors? An unbalanced sector distribution in the input data can bias the portfolio optimizer toward overrepresented sectors.

In [ ]:
sector_ticker = (
    df[['Ticker', 'Sector']].drop_duplicates()
    .groupby('Sector')['Ticker'].count()
    .sort_values(ascending=False)
    .rename('count')
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=sector_ticker, x='count', y='Sector',
            palette='muted', ax=axes[0])
axes[0].set_title('Tickers per GICS Sector')
axes[0].set_xlabel('Number of Tickers')
axes[0].set_ylabel('')
for i, v in enumerate(sector_ticker['count']):
    axes[0].text(v + 0.1, i, str(v), va='center', fontsize=10)

axes[1].pie(sector_ticker['count'], labels=sector_ticker['Sector'],
            autopct='%1.0f%%', startangle=140,
            colors=sns.color_palette('muted', len(sector_ticker)))
axes[1].set_title('Sector Composition of 99-Ticker Universe')

plt.suptitle('S&P 500 Subset \u2014 Ticker Universe by GICS Sector', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/02_sector_distribution.png', bbox_inches='tight')
plt.show()

print(sector_ticker.to_string(index=False))

## 5. Close Price Distribution by Sector

Box plots reveal the price range, median, and outliers for each sector. Note: raw price levels are not comparable across tickers (e.g., NVL vs MSFT) — we use this to understand scale before normalization.

In [ ]:
ticker_stats = (
    df.groupby(['Ticker', 'Sector'])['Close']
    .median()
    .reset_index()
    .rename(columns={'Close': 'MedianClose'})
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=ticker_stats, x='Sector', y='MedianClose',
            palette='muted', ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Median Close Price Distribution by Sector (log scale)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Median Close Price (USD, log)')
axes[0].tick_params(axis='x', rotation=45)

sample_df = df.groupby('Sector').apply(
    lambda x: x.sample(min(200, len(x)), random_state=42)
).reset_index(drop=True)

sns.violinplot(data=sample_df, x='Sector', y='Close',
               palette='muted', inner='quartile', ax=axes[1])
axes[1].set_title('Daily Close Price Violin by Sector (200 samples/sector)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Close Price (USD)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Price Level Distribution Across GICS Sectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/02_price_distribution.png', bbox_inches='tight')
plt.show()

## 6. Macro Covariate Time Series

We plot all 11 macro covariates over the full 5-year period. These covariates are passed as `past_covariates` to Chronos-2 during inference. Understanding their trends and regimes is essential for interpreting forecast outputs — for example, the VIXCLS spike during COVID-19 (2020-03) is a critical contextual signal.

In [ ]:
macro_df = df[df['Ticker'] == df['Ticker'].iloc[0]][['Date'] + COVARIATE_COLS].copy()
macro_df = macro_df.sort_values('Date').set_index('Date')

COVARIATE_LABELS = {
    'DGS10':           '10Y Treasury Yield (%)',
    'VIXCLS':          'VIX (Volatility Index)',
    'UNRATE':          'Unemployment Rate (%)',
    'CPIAUCSL':        'CPI (Index Level)',
    'BAMLH0A0HYM2':    'HY Credit Spread (bps)',
    'T10Y2Y':          '10Y\u20132Y Yield Spread (%)',
    'UMCSENT':         'Consumer Sentiment (Index)',
    'FEDFUNDS_RATE':   'Fed Funds Rate (%)',
    'M2_MONEY_SUPPLY': 'M2 Money Supply ($T)',
    'SOFR':            'SOFR Rate (%)',
    'INFLATION_RATE':  'Inflation Rate (%)',
}

fig, axes = plt.subplots(6, 2, figsize=(16, 22))
axes = axes.flatten()

for i, col in enumerate(COVARIATE_COLS):
    ax = axes[i]
    ax.plot(macro_df.index, macro_df[col], linewidth=1.2, color=sns.color_palette('tab10')[i % 10])
    ax.set_title(COVARIATE_LABELS[col], fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.grid(True, alpha=0.3)

axes[11].set_visible(False)

plt.suptitle('Macro Covariates \u2014 Full 5-Year Timeline  (FRED + Kaggle)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('notebooks/02_macro_timeseries.png', bbox_inches='tight')
plt.show()

## 7. Macro Covariate Correlation Heatmap

Which macro variables move together? High inter-correlation can indicate redundancy (e.g., FEDFUNDS_RATE and SOFR are nearly identical post-2022) and helps us understand which covariates add independent signal to the model.

In [ ]:
corr = macro_df[COVARIATE_COLS].corr()

short_labels = [
    'DGS10', 'VIX', 'UNRATE', 'CPI', 'HY Spread',
    'T10Y2Y', 'Sentiment', 'FedFunds', 'M2', 'SOFR', 'Inflation'
]

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    xticklabels=short_labels,
    yticklabels=short_labels,
    ax=ax,
)
ax.set_title('Macro Covariate Correlation Matrix  (lower triangle)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/02_macro_correlation.png', bbox_inches='tight')
plt.show()

print("High-correlation pairs (|r| > 0.85):")
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
high = corr_pairs[corr_pairs > 0.85].drop_duplicates()
for (a, b), v in high.items():
    print(f"  {a:<22} \u00d7 {b:<22}  r = {v:.3f}")

## 8. Sector-Level Return & Volatility Profile

We compute each ticker's annualised mean return and volatility (using daily log returns), then aggregate by sector. This shows the risk-return landscape that the portfolio optimizer is working with.

In [ ]:
df_sorted = df.sort_values(['Ticker', 'Date'])
df_sorted = df_sorted.copy()
df_sorted['LogReturn'] = df_sorted.groupby('Ticker')['Close'].transform(
    lambda x: np.log(x / x.shift(1))
)

ticker_rv = (
    df_sorted.groupby(['Ticker', 'Sector'])['LogReturn']
    .agg(
        AnnReturn=lambda x: x.mean() * 252 * 100,
        AnnVol=lambda x: x.std() * np.sqrt(252) * 100,
    )
    .reset_index()
    .dropna()
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

palette = sns.color_palette('tab10', n_colors=ticker_rv['Sector'].nunique())
sector_colors = {s: palette[i] for i, s in enumerate(sorted(ticker_rv['Sector'].unique()))}
for sector, grp in ticker_rv.groupby('Sector'):
    axes[0].scatter(grp['AnnVol'], grp['AnnReturn'],
                    label=sector, color=sector_colors[sector],
                    s=60, alpha=0.8)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('Risk\u2013Return Scatter by Ticker (colour = sector)')
axes[0].set_xlabel('Annualised Volatility (%)')
axes[0].set_ylabel('Annualised Return (%)')
axes[0].legend(fontsize=7, ncol=2)

sector_agg = (
    ticker_rv.groupby('Sector')[['AnnReturn', 'AnnVol']]
    .mean()
    .sort_values('AnnReturn', ascending=False)
    .reset_index()
)
x = np.arange(len(sector_agg))
width = 0.38
axes[1].bar(x - width/2, sector_agg['AnnReturn'], width,
            label='Mean Ann. Return (%)', color='steelblue', alpha=0.85)
axes[1].bar(x + width/2, sector_agg['AnnVol'], width,
            label='Mean Ann. Volatility (%)', color='salmon', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sector_agg['Sector'], rotation=40, ha='right', fontsize=9)
axes[1].set_title('Sector-Level Risk\u2013Return Profile')
axes[1].set_ylabel('%')
axes[1].legend()
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')

plt.suptitle('Annualised Return & Volatility \u2014 Ticker & Sector Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/02_risk_return.png', bbox_inches='tight')
plt.show()

## 9. Volume Analysis

Trading volume is a proxy for liquidity and market attention. We compare median daily volume across sectors and plot rolling 30-day volume trends for the top-5 tickers by total volume.

In [ ]:
vol_sector = (
    df.groupby('Sector')['Volume']
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.barplot(data=vol_sector, x='Volume', y='Sector',
            palette='muted', ax=axes[0])
axes[0].set_title('Median Daily Volume by Sector')
axes[0].set_xlabel('Median Daily Volume (shares)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.0f}M'))

top5_vol = (
    df.groupby('Ticker')['Volume'].sum()
    .nlargest(5).index.tolist()
)
for ticker in top5_vol:
    t_df = df[df['Ticker'] == ticker].sort_values('Date')
    rolling_vol = t_df['Volume'].rolling(30).mean()
    axes[1].plot(t_df['Date'], rolling_vol, label=ticker, linewidth=1.5)

axes[1].set_title('Rolling 30-Day Average Volume \u2014 Top 5 Tickers')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume (shares)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.0f}M'))
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Trading Volume Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/02_volume_analysis.png', bbox_inches='tight')
plt.show()

print("Top 5 tickers by total volume:")
for t in top5_vol:
    total = df[df['Ticker'] == t]['Volume'].sum()
    print(f"  {t}: {total/1e9:.2f}B shares total")